# BEGG — Telegram video bot (Google Colab)

Run the cells from top to bottom. This notebook clones or updates the **main** branch, installs dependencies, loads credentials from Colab Secrets, and starts BEGG.

**Required Colab Secrets:** `BOT_TOKEN`, `API_ID`, `API_HASH`, `ADMIN_ID`.

Open **Colab → Secrets** (key icon), add those four values, and enable notebook access. ePorner is the default provider; choose another available source from the Telegram menu. Pornhub is excluded.

In [ ]:
# Check runtime and get the latest BEGG code
!python --version
!git --version

from pathlib import Path
REPO_DIR = Path('/content/Begg')
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    %cd /content/Begg
    !git pull --ff-only origin main
else:
    !git clone --branch main https://github.com/Telebotfaroff/Begg.git /content/Begg
    %cd /content/Begg

print('Repository ready:', REPO_DIR)

In [ ]:
# Install BEGG and provider API dependencies
%cd /content/Begg
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

In [ ]:
# Load private credentials from Google Colab Secrets
from google.colab import userdata
import os

required_secrets = ('BOT_TOKEN', 'API_ID', 'API_HASH', 'ADMIN_ID')
missing = []
for key in required_secrets:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()

if missing:
    raise RuntimeError('Missing or inaccessible Colab Secrets: ' + ', '.join(missing) + '. Add them in the Secrets panel and enable notebook access, then rerun this cell.')

for key in ('API_ID', 'ADMIN_ID'):
    try:
        int(os.environ[key])
    except ValueError as exc:
        raise ValueError(f'{key} must contain a numeric value.') from exc

# Optional settings
os.environ.setdefault('DOWNLOAD_CONCURRENCY', '5')
os.environ.setdefault('UPLOAD_CONCURRENCY', '2')
os.environ.setdefault('SEARCH_LIMIT', '10')
os.environ.setdefault('UPDATE_INTERVAL', '3')
os.environ.setdefault('VIDEO_QUALITY', 'best')
print('Required secrets loaded. Values are not displayed.')

In [ ]:
# Start BEGG — keep this cell running while you use the bot
%cd /content/Begg
!python bot.py

## Updating later

Stop the running bot, rerun the repository update cell, rerun the dependency installation cell, then rerun the secrets cell and start cell. Keep the Colab runtime active while the bot is running; stopping or disconnecting the runtime stops the bot.